# Engenharia de Features - Parte 1
## Fundamentos e Transformações Básicas

**Disciplina:** INF1032 - Introdução à Ciência de Dados  
**Professor:** Adriano Branco  
**Instituição:** PUC-Rio - Departamento de Informática  

### Aula 09 - Parte 1 de 2

---


# Parte I: Conceitos Teóricos

# ==================================================
# 1. INTRODUÇÃO À ENGENHARIA DE FEATURES
# ==================================================

## 🎯 **Objetivos da Aula - Parte 1**

Ao final desta parte, você será capaz de:

1. **Compreender** a importância da engenharia de features em projetos de ciência de dados
2. **Realizar análise exploratória** para identificar oportunidades de transformação
3. **Aplicar transformações matemáticas básicas** (log, raiz, potência, inverso)
4. **Avaliar o impacto** das transformações usando métricas como correlação e R²
5. **Identificar relações não-lineares** nos dados e linearizá-las
6. **Interpretar visualizações** para escolher as melhores transformações

---

In [ ]:
# @title
# Importação das bibliotecas
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import (
    StandardScaler,
    PolynomialFeatures,
    FunctionTransformer
)
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_selection import SelectKBest, f_regression, RFE
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Configurações de visualização
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('viridis')
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)

# ==================================================
# 2. O QUE É ENGENHARIA DE FEATURES E POR QUE É IMPORTANTE?
# ==================================================

### DEFINIÇÃO E IMPORTÂNCIA

**Engenharia de Features** é o processo de utilizar conhecimento de domínio para extrair características (features) dos dados brutos que ajudam os algoritmos de aprendizado de máquina a funcionar melhor. Este processo envolve a transformação dos dados brutos em um formato que melhor representa o problema subjacente para os modelos preditivos.

**Por que a engenharia de features é tão importante?**
1. **Melhora o desempenho dos modelos**: Boas features podem simplificar e destacar os padrões, permitindo que algoritmos mais simples funcionem bem.
   
2. **Captura conhecimento de domínio**: Incorpora o entendimento do especialista sobre o problema, tornando o modelo mais interpretável e confiável.
   
3. **Reduz a necessidade de complexidade do modelo**: Com features bem projetadas, modelos mais simples podem superar modelos complexos, economizando tempo de processamento e recursos.
   
4. **Lida com dados imperfeitos**: Ajuda a superar limitações dos dados brutos, como ruído, valores ausentes e relações não-lineares.
   
5. **Expande a representação do problema**: Traz à tona informações latentes que não estão explícitas nos dados originais.

A importância da engenharia de features é frequentemente sintetizada na frase:

> "A feature engineering is the art part of data science." - Andrew Ng  
 *(A engenharia de features é a parte artística da ciência de dados.)*

Os modelos de machine learning são apenas tão bons quanto os dados que recebem. Boas features são frequentemente mais importantes que algoritmos sofisticados. Como exemplo, um modelo linear simples com features bem projetadas frequentemente supera modelos complexos com features ruins ou limitadas.

### O ESPECTRO DA ENGENHARIA DE FEATURES

A engenharia de features abrange um amplo espectro de técnicas, desde as mais simples até as mais complexas:

1. **Transformações Simples:**
   - Normalização e padronização
   - Transformações logarítmicas, raiz quadrada
   - Binarização
   - Codificação de variáveis categóricas

2. **Criação de Features:**
   - Combinações de features existentes
   - Agregações (média, soma, mínimo, máximo)
   - Features polinomiais e interações
   - Decomposição de atributos complexos (datas, textos, imagens)

3. **Extração de Features:**
   - Redução de dimensionalidade (PCA, t-SNE)
   - Extração de características de séries temporais
   - Processamento de texto e NLP
   - Processamento de imagens e sinais

4. **Seleção de Features:**
   - Métodos baseados em filtro (correlação, chi-quadrado)
   - Métodos baseados em wrapper (recursive feature elimination)
   - Métodos baseados em importância (feature importance de árvores)
   - Seleção baseada em regularização (Lasso)

# ==================================================
# 3. CRIANDO UM CONJUNTO DE DADOS DE EXEMPLO
# ==================================================

Para explorar técnicas de engenharia de features, vamos criar um conjunto de dados sintético que simula dados de imóveis em uma cidade fictícia. Estes dados terão características que se beneficiarão de diferentes técnicas de engenharia de features.

In [ ]:
# @title
# Definindo o seed para reprodutibilidade
np.random.seed(42)

# Número de amostras
n = 1000

# Gerando coordenadas para imóveis (latitude e longitude)
# Centrado aproximadamente em uma cidade fictícia
base_latitude = 40.7
base_longitude = -74.0
latitude = base_latitude + np.random.normal(0, 0.1, n)
longitude = base_longitude + np.random.normal(0, 0.1, n)

# Criando três centros na cidade (CBD - Central Business District)
centers = [
    (base_latitude + 0.05, base_longitude + 0.03),  # Centro financeiro
    (base_latitude - 0.02, base_longitude - 0.05),  # Centro de entretenimento
    (base_latitude - 0.08, base_longitude + 0.08)   # Centro histórico
]

# Calculando a distância para cada centro
distances = []
for center_lat, center_lon in centers:
    # Distância euclidiana como aproximação simples
    dist = np.sqrt((latitude - center_lat)**2 + (longitude - center_lon)**2)
    # Convertendo para km (aproximadamente) usando um fator de escala simples
    dist = dist * 111  # 1 grau = aprox. 111 km
    distances.append(dist)

# Gerando características básicas do imóvel
area_m2 = np.random.lognormal(mean=4.5, sigma=0.3, size=n)  # Área em m²
quartos = np.random.choice([1, 2, 3, 4, 5, 6], size=n, p=[0.1, 0.2, 0.35, 0.2, 0.1, 0.05])
banheiros = np.random.choice([1, 2, 3, 4], size=n, p=[0.3, 0.4, 0.25, 0.05])
idade_anos = np.random.gamma(shape=3, scale=10, size=n)  # Idade do imóvel
garagem_vagas = np.random.choice([0, 1, 2, 3], size=n, p=[0.2, 0.4, 0.3, 0.1])

# Gerando qualidade do imóvel (1-10)
qualidade_material = np.random.choice(range(1, 11), size=n, p=[0.02, 0.05, 0.08, 0.1, 0.15, 0.2, 0.15, 0.1, 0.08, 0.07])
qualidade_acabamento = np.random.choice(range(1, 11), size=n, p=[0.03, 0.04, 0.08, 0.1, 0.15, 0.2, 0.15, 0.1, 0.08, 0.07])

# Características categóricas
estilo_arquitetonico = np.random.choice(
    ['Moderno', 'Clássico', 'Contemporâneo', 'Colonial', 'Industrial'],
    size=n,
    p=[0.3, 0.2, 0.25, 0.15, 0.1]
)

tipo_imovel = np.random.choice(
    ['Apartamento', 'Casa', 'Sobrado', 'Loft', 'Cobertura'],
    size=n,
    p=[0.45, 0.25, 0.15, 0.1, 0.05]
)

# Gerando datas de construção e renovação (se houver)
data_atual = datetime(2023, 1, 1)
datas_construcao = [data_atual - timedelta(days=int(365 * idade)) for idade in idade_anos]

# Alguns imóveis foram renovados
renovado = np.random.choice([0, 1], size=n, p=[0.6, 0.4])
anos_desde_renovacao = np.zeros(n)
for i in range(n):
    if renovado[i] == 1:
        # Renovação ocorreu em algum momento entre a construção e agora
        max_anos = max(1, idade_anos[i] - 1)  # Pelo menos 1 ano após construção
        anos_desde_renovacao[i] = np.random.uniform(0, max_anos)
    else:
        anos_desde_renovacao[i] = np.nan  # Não renovado

# Features adicionais
andar = np.zeros(n, dtype=int)  # Apenas para apartamentos
for i in range(n):
    if tipo_imovel[i] in ['Apartamento', 'Loft', 'Cobertura']:
        probabilidades = [0.05, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.05, 0.02, 0.02, 0.02, 0.01, 0.01, 0.005, 0.005, 0.003, 0.002, 0.005]  # O último valor foi ajustado para 0.005
        andar[i] = np.random.choice(range(1, 21), p=probabilidades)
    else:
        andar[i] = 0  # Térreo para casas e sobrados

# Características de localização e amenidades
metro_distancia = np.random.lognormal(mean=0, sigma=1, size=n) * 500  # Em metros
parque_distancia = np.random.lognormal(mean=0.5, sigma=1, size=n) * 400  # Em metros
escola_distancia = np.random.lognormal(mean=0.5, sigma=0.8, size=n) * 300  # Em metros

# Algumas características binárias
tem_piscina = np.random.choice([0, 1], size=n, p=[0.8, 0.2])
tem_academia = np.random.choice([0, 1], size=n, p=[0.7, 0.3])
tem_condominio = np.random.choice([0, 1], size=n, p=[0.4, 0.6])
vista_para_agua = np.random.choice([0, 1], size=n, p=[0.9, 0.1])

# Gerar o preço do imóvel (variável alvo) baseado nas características
# Vamos criar algumas relações não-lineares e interações complexas

# Base do preço (R$ 400.000,00)
preco_base = 400000

# Componentes do preço com relações não-lineares
componente_area = 2000 * np.sqrt(area_m2)  # Relação não-linear com área
componente_quartos = 50000 * np.log1p(quartos)  # Mais quartos = mais valor, mas com retorno diminuído
componente_banheiros = 40000 * banheiros  # Relação linear com número de banheiros
componente_idade = -20000 * np.log1p(idade_anos)  # Mais velho = menos valor, de forma não-linear
componente_distancia = -100000 * np.log1p(np.min(distances, axis=0))  # Quanto mais perto do centro, melhor
componente_garagem = 30000 * garagem_vagas  # Valor linear por vaga de garagem
componente_qualidade = 10000 * (qualidade_material + qualidade_acabamento)  # Valor linear por qualidade

# Componentes categóricos
componente_tipo = np.zeros(n)
tipo_valores = {
    'Apartamento': 0,
    'Casa': 50000,
    'Sobrado': 70000,
    'Loft': 30000,
    'Cobertura': 150000
}
for i, tipo in enumerate(tipo_imovel):
    componente_tipo[i] = tipo_valores[tipo]

# Componentes para características binárias
componente_piscina = 50000 * tem_piscina
componente_academia = 20000 * tem_academia
componente_vista = 80000 * vista_para_agua

# Interações (exemplos de features que interagem entre si)
# Apartamentos em andares mais altos são mais valorizados
interacao_andar_tipo = np.zeros(n)
for i in range(n):
    if tipo_imovel[i] in ['Apartamento', 'Loft']:
        interacao_andar_tipo[i] = 5000 * andar[i]
    elif tipo_imovel[i] == 'Cobertura':
        interacao_andar_tipo[i] = 8000 * andar[i]  # Coberturas em andares altos são ainda mais valorizadas

# Casas maiores com piscina são mais valorizadas que apartamentos com piscina
interacao_piscina_tipo = np.zeros(n)
for i in range(n):
    if tem_piscina[i] == 1:
        if tipo_imovel[i] in ['Casa', 'Sobrado']:
            interacao_piscina_tipo[i] = 30000  # Bônus adicional para casas com piscina

# Renovação recente aumenta o valor, especialmente para imóveis mais antigos
componente_renovacao = np.zeros(n)
for i in range(n):
    if not np.isnan(anos_desde_renovacao[i]):  # Se foi renovado
        # Quanto mais recente a renovação, maior o valor
        tempo_desde_renovacao = anos_desde_renovacao[i]
        # Bônus depende da idade do imóvel e da antiguidade da renovação
        componente_renovacao[i] = 50000 * np.exp(-0.1 * tempo_desde_renovacao) * np.log1p(idade_anos[i])

# Adicionar ruído aleatório (R$ ±50.000,00)
ruido = np.random.normal(0, 50000, n)

# Calcular o preço final
preco = (preco_base +
         componente_area +
         componente_quartos +
         componente_banheiros +
         componente_idade +
         componente_distancia +
         componente_garagem +
         componente_qualidade +
         componente_tipo +
         componente_piscina +
         componente_academia +
         componente_vista +
         interacao_andar_tipo +
         interacao_piscina_tipo +
         componente_renovacao +
         ruido)

# Garantir que todos os preços sejam positivos
preco = np.maximum(preco, 100000)

In [ ]:
# @title
# Criando o DataFrame final
imoveis = pd.DataFrame({
    'latitude': latitude,
    'longitude': longitude,
    'area_m2': area_m2,
    'quartos': quartos,
    'banheiros': banheiros,
    'idade_anos': idade_anos,
    'garagem_vagas': garagem_vagas,
    'qualidade_material': qualidade_material,
    'qualidade_acabamento': qualidade_acabamento,
    'estilo_arquitetonico': estilo_arquitetonico,
    'tipo_imovel': tipo_imovel,
    'data_construcao': datas_construcao,
    'renovado': renovado,
    'anos_desde_renovacao': anos_desde_renovacao,
    'andar': andar,
    'distancia_metro': metro_distancia,
    'distancia_parque': parque_distancia,
    'distancia_escola': escola_distancia,
    'tem_piscina': tem_piscina,
    'tem_academia': tem_academia,
    'tem_condominio': tem_condominio,
    'vista_para_agua': vista_para_agua,
    'distancia_centro1': distances[0],  # Distância para o centro financeiro
    'distancia_centro2': distances[1],  # Distância para o centro de entretenimento
    'distancia_centro3': distances[2],  # Distância para o centro histórico
    'preco': preco
})

# Arredondando algumas colunas para facilitar a visualização
imoveis['area_m2'] = np.round(imoveis['area_m2'], 2)
imoveis['preco'] = np.round(imoveis['preco'], 2)
imoveis['idade_anos'] = np.round(imoveis['idade_anos'], 1)
imoveis['anos_desde_renovacao'] = imoveis['anos_desde_renovacao'].round(1)

# Exibindo as primeiras linhas
imoveis.head()

In [ ]:
# @title
# Verificando informações básicas do dataset
print("Informações do Dataset:")
print(f"Número de registros: {len(imoveis)}")
print(f"Número de colunas: {len(imoveis.columns)}")
print("\nTipos de dados:")
print(imoveis.dtypes.value_counts())

# Estatísticas descritivas para colunas numéricas
print("\nEstatísticas Descritivas (colunas numéricas):")
print(imoveis.describe())

In [ ]:
# @title
# Análise rápida das variáveis categóricas
print("Distribuição das variáveis categóricas:")
for col in ['estilo_arquitetonico', 'tipo_imovel']:
    print(f"\n{col}:")
    print(imoveis[col].value_counts(normalize=True) * 100)

In [ ]:
# @title
# Verificando valores ausentes
missing_values = imoveis.isnull().sum()
missing_values_sem_renovacao = missing_values.drop('anos_desde_renovacao', errors='ignore')
missing_values_sem_renovacao = missing_values_sem_renovacao[missing_values_sem_renovacao > 0]

if len(missing_values_sem_renovacao) > 0:
    print("Colunas com valores ausentes (exceto anos_desde_renovacao):")
    print(missing_values_sem_renovacao)
else:
    print(f"Não há valores ausentes, exceto anos_desde_renovacao ({missing_values.get('anos_desde_renovacao', 0)} valores) para imóveis não renovados.")

In [ ]:
# @title
# Visualizando a distribuição da variável alvo (preço) e testando normalidade
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(18, 5))

# 1. Histograma com curva normal sobreposta
ax1 = axes[0]
sns.histplot(imoveis['preco'], kde=True, stat='density', ax=ax1)
# Sobrepor distribuição normal teórica
mu, std = imoveis['preco'].mean(), imoveis['preco'].std()
x = np.linspace(imoveis['preco'].min(), imoveis['preco'].max(), 100)
ax1.plot(x, stats.norm.pdf(x, mu, std), 'r-', linewidth=2, label='Normal teórica')
ax1.set_title('Distribuição dos Preços dos Imóveis')
ax1.set_xlabel('Preço (R$)')
ax1.set_ylabel('Densidade')
ax1.legend()

# 2. Q-Q Plot (Quantile-Quantile)
ax2 = axes[1]
stats.probplot(imoveis['preco'], dist="norm", plot=ax2)
ax2.set_title('Q-Q Plot\n(Pontos na linha = distribuição normal)')
ax2.get_lines()[1].set_color('red')


# Teste de Shapiro-Wilk (funciona bem para amostras < 5000)
if len(imoveis) < 5000:
    statistic, p_value = stats.shapiro(imoveis['preco'])
    teste_nome = "Shapiro-Wilk"
else:
    # Para amostras grandes, usar Kolmogorov-Smirnov
    statistic, p_value = stats.normaltest(imoveis['preco'])
    teste_nome = "D'Agostino-Pearson"

# Calcular assimetria e curtose
skewness = stats.skew(imoveis['preco'])
kurtosis = stats.kurtosis(imoveis['preco'])

# Exibir métricas
info_text = f"""📊 Análise de Normalidade

Teste {teste_nome}:
• Estatística: {statistic:.4f}
• p-valor: {p_value:.4f}
• Conclusão: {'Não normal' if p_value < 0.05 else 'Normal'} (α=0.05)

Métricas de Forma:
• Assimetria: {skewness:.3f}
(0 = simétrica, >0 = cauda direita, <0 = cauda esquerda)

• Curtose: {kurtosis:.3f}
(0 = normal, >0 = leptocúrtica, <0 = platicúrtica)

Interpretação:
• Assimetria ~0 e Curtose ~0 = Normal
• |Assimetria| < 0.5 = Aproximadamente simétrica
• |Curtose| < 3 = Aproximadamente mesocúrtica
"""


plt.suptitle('Análise de Normalidade da Distribuição dos Preços',fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(info_text)

# ==================================================
# 4. ANÁLISE EXPLORATÓRIA PARA ENGENHARIA DE FEATURES
# ==================================================

A análise exploratória de dados (EDA) é uma etapa crucial para identificar oportunidades de engenharia de features. Antes de criar novas features, precisamos entender profundamente os dados e as relações entre variáveis.

In [ ]:
# @title Analisando a correlação entre as variáveis numéricas e o preço
numeric_cols = imoveis.select_dtypes(include=['int64', 'float64']).columns.tolist()
corr = imoveis[numeric_cols].corr()

# Ordenando as correlações com o preço
price_corr = corr['preco'].sort_values(ascending=False)
print("Correlação com o preço:")
print(price_corr)

In [ ]:
# @title Visualizando a matriz de correlação
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Máscara para o triângulo superior
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Matriz de Correlação das Variáveis Numéricas')
plt.tight_layout()
plt.show()

In [ ]:
# @title Explorando relações com variáveis categóricas
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Preço por tipo de imóvel
sns.boxplot(x='tipo_imovel', y='preco', data=imoveis, ax=axes[0])
axes[0].set_title('Preço por Tipo de Imóvel')
axes[0].set_xlabel('Tipo de Imóvel')
axes[0].set_ylabel('Preço (R$)')
axes[0].tick_params(axis='x', rotation=45)

# Preço por estilo arquitetônico
sns.boxplot(x='estilo_arquitetonico', y='preco', data=imoveis, ax=axes[1])
axes[1].set_title('Preço por Estilo Arquitetônico')
axes[1].set_xlabel('Estilo Arquitetônico')
axes[1].set_ylabel('Preço (R$)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# @title Analisando a relação entre variáveis numéricas e o preço
# Selecionando algumas variáveis importantes para visualização
features_to_plot = ['area_m2', 'quartos', 'banheiros', 'idade_anos',
                    'qualidade_material', 'andar', 'distancia_centro1','area_m2']

fig, axes = plt.subplots(3, 3, figsize=(18, 12))
axes = axes.flatten()

for i, feature in enumerate(features_to_plot):
    if i < len(axes):
        sns.scatterplot(x=feature, y='preco', data=imoveis, alpha=0.5, ax=axes[i])
        axes[i].set_title(f'Preço vs {feature}')
        axes[i].set_xlabel(feature)
        axes[i].set_ylabel('Preço (R$)')

# Relação entre distância do metro e preço
if len(features_to_plot) < len(axes):
    sns.scatterplot(x='distancia_metro', y='preco', data=imoveis, alpha=0.5, ax=axes[len(features_to_plot)])
    axes[len(features_to_plot)].set_title('Preço vs Distância ao Metrô')
    axes[len(features_to_plot)].set_xlabel('Distância ao Metrô (m)')
    axes[len(features_to_plot)].set_ylabel('Preço (R$)')

# Removendo subplots vazios
for i in range(len(features_to_plot) + 1, len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# @title Analisando o impacto das características binárias
binary_features = ['tem_piscina', 'tem_academia', 'tem_condominio', 'vista_para_agua', 'renovado']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feature in enumerate(binary_features):
    if i < len(axes):
        sns.boxplot(x=feature, y='preco', data=imoveis, ax=axes[i])
        axes[i].set_title(f'Preço vs {feature}')
        axes[i].set_xlabel(feature + ' (0=Não, 1=Sim)')
        axes[i].set_ylabel('Preço (R$)')

        # Calculando e exibindo a média
        means = imoveis.groupby(feature)['preco'].mean()
        for j, mean_val in enumerate(means):
            axes[i].text(j, mean_val + 50000, f'Média: {mean_val:.0f}',
                      ha='center', va='center', color='red', fontweight='bold')

# Removendo subplots vazios
for i in range(len(binary_features), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Analisando possíveis interações entre variáveis
# Exemplo: relação entre tipo de imóvel e andar
plt.figure(figsize=(14, 6))

# Filtrando apenas tipos relevantes para andar
tipos_relevantes = ['Apartamento', 'Loft', 'Cobertura']
df_filtrado = imoveis[imoveis['tipo_imovel'].isin(tipos_relevantes)]

# Visualizando a relação
sns.scatterplot(x='andar', y='preco', hue='tipo_imovel', data=df_filtrado, s=80, alpha=0.7)
plt.title('Relação entre Andar, Tipo de Imóvel e Preço')
plt.xlabel('Andar')
plt.ylabel('Preço (R$)')
plt.legend(title='Tipo de Imóvel')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# @title
# Analisando outra interação: relação entre tipo de imóvel e ter piscina
plt.figure(figsize=(14, 6))

# Calculando o preço médio por tipo de imóvel e presença de piscina
pivot_data = imoveis.pivot_table(values='preco', index='tipo_imovel', columns='tem_piscina', aggfunc='mean')
pivot_data.columns = ['Sem Piscina', 'Com Piscina']

# Visualizando
ax = pivot_data.plot(kind='bar', figsize=(14, 6))
plt.title('Preço Médio por Tipo de Imóvel e Presença de Piscina')
plt.xlabel('Tipo de Imóvel')
plt.ylabel('Preço Médio (R$)')
plt.xticks(rotation=45)
plt.grid(axis='y', alpha=0.3)

# Adicionando os valores nas barras
for i, container in enumerate(ax.containers):
    ax.bar_label(container, fmt='%.0f', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# @title
# Visualizando a distribuição espacial dos imóveis e seus preços
plt.figure(figsize=(14, 8))

# Scatter plot dos imóveis
scatter = plt.scatter(imoveis['longitude'], imoveis['latitude'],
                      c=imoveis['preco'], cmap='plasma',
                      s=50, alpha=0.7, edgecolors='k', linewidths=0.5)

# Marcando os centros da cidade
plt.scatter([center[1] for center in centers], [center[0] for center in centers],
           color='red', s=200, marker='*', edgecolors='black', label='Centros da Cidade')

# Adicionando anotações para os centros
center_names = ['Centro Financeiro', 'Centro de Entretenimento', 'Centro Histórico']
for i, (lon, lat) in enumerate(zip([center[1] for center in centers], [center[0] for center in centers])):
    plt.annotate(center_names[i], (lon, lat), xytext=(10, 10), textcoords='offset points',
                fontsize=12, fontweight='bold')

plt.colorbar(scatter, label='Preço (R$)')
plt.title('Distribuição Espacial dos Imóveis e seus Preços')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

### Conclusões da Análise Exploratória

Através da análise exploratória, identificamos diversas oportunidades para engenharia de features:

1. **Relações não-lineares**:
   - A idade do imóvel tem relação não-linear com o preço (decaimento
  logarítmico)
   - A distância aos centros e amenidades também apresenta comportamento não-linear

2. **Interações importantes**:
   - Forte interação entre tipo de imóvel e andar (coberturas em andares altos são mais valorizadas)
   - Interação entre tipo de imóvel e presença de piscina (mais valioso para casas do que apartamentos)
   - Interação entre idade e renovação

3. **Variáveis categóricas**:
   - Tipo de imóvel têm forte impacto no preço
   - Necessitam de codificação adequada

4. **Características geoespaciais**:
   - Localização influencia significativamente o preço
   - Distância aos centros e amenidades pode ser transformada

5. **Variáveis temporais**:
   - Idade do imóvel e tempo desde a renovação podem ser explorados de forma conjunta

6. **Agregações potenciais**:
   - Combinar qualidade do material e acabamento
   - Criar índice de amenidades (piscina, academia, etc.)
   - Agregar distâncias para diferentes pontos de interesse

Estas observações nos guiarão na criação de novas features nas próximas seções.

# ==================================================
# 5. TRANSFORMAÇÕES MATEMÁTICAS BÁSICAS
# ==================================================

As transformações matemáticas básicas são frequentemente o primeiro passo na engenharia de features. Elas podem ajudar a linearizar relações, normalizar distribuições e destacar padrões importantes nos dados.

Principais transformações matemáticas incluem:

1. **Transformações logarítmicas**: log(x), log1p(x) = log(1+x)
2. **Transformações de potência**: x², x³, √x, ∛x
3. **Transformações exponenciais**: eˣ, 10ˣ
4. **Transformações trigonométricas**: sin(x), cos(x), tan(x)
5. **Transformações recíprocas**: 1/x

Vamos implementar algumas destas transformações nas features identificadas durante a análise exploratória:

### 📊 O que significa R² (Coeficiente de Determinação)?

  O **R²** mede quanto da variação no preço é explicada pela variável
  analisada. É uma métrica fundamental para avaliar a qualidade do ajuste de
   um modelo de regressão.

  #### Interpretação dos valores:
  - **R² = 0.00**: A variável não explica NADA da variação do preço
  - **R² = 0.25**: A variável explica 25% da variação do preço
  - **R² = 0.50**: A variável explica 50% da variação do preço
  - **R² = 1.00**: A variável explica TODA a variação do preço



In [ ]:
# @title Criando um exemplo demonstrativo para mostrar o poder das transformações
np.random.seed(42)

# Gerando dados com relação exponencial clara
x_demo = np.random.uniform(1, 10, 200)
# y tem relação exponencial com x (então log(y) será linear com x)
y_demo = 100 * np.exp(0.5 * x_demo) + np.random.normal(0, 50, 200)

# Criando DataFrame
df_demo = pd.DataFrame({
    'x': x_demo,
    'x_log': np.log(x_demo),
    'x_squared': x_demo ** 2,
    'x_sqrt': np.sqrt(x_demo),
    'y': y_demo
})

# Visualizando o efeito das transformações
feature_to_explore = 'x'
transformations = ['x', 'x_log', 'x_squared', 'x_sqrt']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, transform in enumerate(transformations):
    sns.scatterplot(x=transform, y='y', data=df_demo, alpha=0.5,ax=axes[i])

    # Adicionando linha de tendência
    x_vals = df_demo[transform].values.reshape(-1, 1)
    y_vals = df_demo['y'].values
    model = LinearRegression().fit(x_vals, y_vals)
    x_range = np.linspace(min(x_vals), max(x_vals), 100).reshape(-1, 1)
    y_pred = model.predict(x_range)
    axes[i].plot(x_range, y_pred, color='red', linewidth=2)

    # Adicionando R²
    r2 = r2_score(y_vals, model.predict(x_vals))
    axes[i].text(0.05, 0.95, f'R² = {r2:.3f}',transform=axes[i].transAxes,
                fontsize=12, fontweight='bold', verticalalignment='top')

    axes[i].set_title(f'y vs {transform}')
    axes[i].set_xlabel(transform)
    axes[i].set_ylabel('y')

plt.suptitle('Exemplo Demonstrativo: Como Transformações Podem Linearizar Relações', fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# @title
# Criar uma cópia do DataFrame para adicionar as transformações
imoveis_transformed = imoveis.copy()

# Primeiro, criar a distância mínima aos centros
imoveis_transformed['distancia_min_centros'] = imoveis_transformed[
    ['distancia_centro1', 'distancia_centro2', 'distancia_centro3']].min(axis=1)

# Criar transformações para distância mínima
imoveis_transformed['distancia_min_centros_log'] = np.log1p(imoveis_transformed['distancia_min_centros'])
imoveis_transformed['distancia_min_centros_sqrt'] = np.sqrt(imoveis_transformed['distancia_min_centros'])
imoveis_transformed['distancia_min_centros_inv'] = 1 / (imoveis_transformed['distancia_min_centros'] + 1)

# Visualizando o efeito das transformações
feature_to_explore = 'distancia_min_centros'
transformations = [feature_to_explore, f'{feature_to_explore}_log',
                   f'{feature_to_explore}_sqrt', f'{feature_to_explore}_inv']

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

for i, transform in enumerate(transformations):
    sns.scatterplot(x=transform, y='preco', data=imoveis_transformed, alpha=0.5, ax=axes[i])

    # Adicionando linha de tendência
    x = imoveis_transformed[transform].values.reshape(-1, 1)
    y = imoveis_transformed['preco'].values
    model = LinearRegression().fit(x, y)
    x_range = np.linspace(min(x), max(x), 100).reshape(-1, 1)
    y_pred = model.predict(x_range)
    axes[i].plot(x_range, y_pred, color='red', linewidth=2)

    # Adicionando R²
    r2 = r2_score(y, model.predict(x))
    axes[i].text(0.05, 0.95, f'R² = {r2:.3f}', transform=axes[i].transAxes,
                fontsize=12, fontweight='bold', verticalalignment='top')

    axes[i].set_title(f'Preço vs {transform}')
    axes[i].set_xlabel(transform)
    axes[i].set_ylabel('Preço (R$)')

plt.tight_layout()
plt.show()

  #### No nosso exemplo:
  - **R² = 0.225** → A distância original explica 22.5% da variação dos
  preços
  - **R² = 0.243** → Após transformação log, explica 24.3% da variação
  - **Melhoria**: Aumento de 1.8 pontos percentuais (ou 8% de melhoria
  relativa)


  > 💡 **Importante**: Em datasets de imóveis, muitos fatores influenciam o
  preço simultaneamente (localização, tamanho, idade, amenidades, etc.). Por
   isso, uma única variável com R² = 0.24 já é considerada relevante! Em
  contextos multivariados complexos, R² baixos para variáveis individuais
  são esperados e normais.


In [ ]:
# @title Analisando o impacto das transformações em UMA variável
dist_col = 'distancia_min_centros'
transformacoes = {
    'Original': imoveis_transformed[dist_col],
    'Log': imoveis_transformed[f'{dist_col}_log'],
    'Raiz Quadrada': imoveis_transformed[f'{dist_col}_sqrt'],
    'Inversa (1/x)': imoveis_transformed[f'{dist_col}_inv']
}

# Calculando correlações
correlacoes = {}
for nome, valores in transformacoes.items():
    correlacoes[nome] = valores.corr(imoveis_transformed['preco'])

# Visualização mais clara
plt.figure(figsize=(10, 6))
cores = ['red' if corr < 0 else 'green' for corr in correlacoes.values()]
bars = plt.barh(list(correlacoes.keys()), list(correlacoes.values()),color=cores)
plt.xlabel('Correlação com Preço')
plt.title('Impacto das Transformações na Correlação - distancia_min_centros x preço \n(Vermelho = Negativa,Verde = Positiva)')
plt.axvline(x=0, color='black', linestyle='-', alpha=0.3)
plt.grid(axis='x', alpha=0.3)

# Adicionar valores nas barras
for bar, (nome, corr) in zip(bars, correlacoes.items()):
    plt.text(corr, bar.get_y() + bar.get_height()/2, f'{corr:.3f}',
            ha='left' if corr > 0 else 'right', va='center')

plt.tight_layout()
plt.show()

### Análise das Transformações

Transformações matemáticas podem ter um impacto significativo na relação entre as variáveis:

1. **Transformações logarítmicas** (log, log1p):
   - Úteis para comprimir valores altos e expandir valores baixos
   - Linearizam relações exponenciais (ex: área vs preço)
   - Tratam distribuições assimétricas positivas (skewed right)
   - Especialmente úteis para variáveis como área, preço, distância

2. **Transformações de potência** (quadrado, raiz quadrada):
   - Quadrado: amplifica diferenças em valores altos
   - Raiz quadrada: comprime valores altos e expande valores baixos (menos agressiva que log)
   - Útil para relações que crescem ou diminuem em taxas variáveis

3. **Transformações inversas** (1/x):
   - Transformam distâncias em "proximidades" (quanto maior a distância, menor o valor)
   - Úteis quando o inverso tem significado intuitivo
   - Muito eficaz para variáveis de distância, onde a proximidade é mais relevante que a distância

Observamos que para variáveis espaciais, as transformações inversas frequentemente mostram maior correlação com o preço, o que faz sentido intuitivamente: a "proximidade" a pontos de interesse é mais diretamente relacionada ao valor do imóvel do que a "distância".

---

## 🎯 **Micro-atividades Práticas**

---

## 🔍 Micro-atividade 1: EDA Detective (10min)

**Objetivo**: Identificar oportunidades de feature engineering através de análise exploratória sistemática

**Cenário**: Você recebeu um novo dataset e precisa identificar as melhores oportunidades para engenharia de features antes de construir um modelo preditivo.

**Sua tarefa**:
1. Carregue o dataset tips do seaborn: `df = sns.load_dataset('tips')`
2. Identifique 3 oportunidades de feature engineering analisando:
   - **Relações não-lineares**: Plote scatter plots e identifique variáveis que podem se beneficiar de transformações
   - **Interações potenciais**: Analise como 2+ variáveis podem interagir (ex: day + time)
   - **Agregações possíveis**: Identifique variáveis que podem ser combinadas em índices ou ratios
3. Para cada oportunidade identificada, justifique:
   - Por que essa transformação faria sentido?
   - Qual seria o benefício esperado para o modelo?
   - Que tipo de transformação você aplicaria?

**Dicas**:
- Use `df.describe()`, `df.corr()`, e visualizações (scatter, box plots)
- Pense no contexto de negócio: o que faria sentido em um restaurante?
- Considere ratios como `tip/total_bill`, interações como `size*day`, etc.

**Bonus**: Crie pelo menos 1 nova feature baseada em suas descobertas e compare a correlação com o target antes e depois

In [ ]:
# 🔍 SEU CÓDIGO AQUI - EDA Detective
# Identifique oportunidades de feature engineering no dataset tips

# 1. Carregar e explorar o dataset
# 2. Analisar correlações e distribuições
# 3. Identificar 3 oportunidades de FE
# 4. Justificar cada oportunidade

# Estrutura sugerida para suas descobertas:
# oportunidades = {
#     'oportunidade_1': {
#         'tipo': 'transformacao_nao_linear',  # ou 'interacao' ou 'agregacao'
#         'variaveis': ['variavel1', 'variavel2'],
#         'transformacao_sugerida': 'log(x), x*y, ratio, etc',
#         'justificativa': 'Por que faria sentido...',
#         'beneficio_esperado': 'Melhor captura de...'
#     },
#     # ... mais oportunidades
# }

import seaborn as sns
df = sns.load_dataset('tips')

print("Implemente sua análise aqui!")
print("Tempo estimado: 10 minutos")

## 🎯 Micro-atividade 2: Transformation Lab (8min)

**Objetivo**: Aplicar transformações matemáticas para melhorar a relação linear entre features e target

**Cenário**: Você identificou que algumas variáveis têm relação não-linear com o target, prejudicando a performance de modelos lineares. Precisa encontrar as melhores transformações.

**Sua tarefa**:
1. Use os dados dos imóveis criados anteriormente: `imoveis_transformed`
2. Teste **4 transformações diferentes** na variável `area_m2`:
   - Original (sem transformação)
   - Logarítmica: `np.log1p(area_m2)`
   - Raiz quadrada: `np.sqrt(area_m2)`  
   - Quadrática: `area_m2 ** 2`
3. Para cada transformação:
   - Calcule a correlação com o preço
   - Fit um modelo Linear Regression simples
   - Calcule o R² score
   - Visualize o scatter plot com linha de regressão
4. Determine qual transformação funciona melhor e explique por quê

**Dicas**:
- Use `scipy.stats.pearsonr()` ou `df.corr()` para correlações
- Use `sklearn.linear_model.LinearRegression` para modelos simples
- Use `sklearn.metrics.r2_score` para avaliar performance
- Considere qual transformação lineariza melhor a relação

**Bonus**: Teste o mesmo processo com uma variável de distância (ex: `distancia_metro`) e compare os resultados

In [ ]:
# ⚡ SEU CÓDIGO AQUI - Transformation Lab
# Teste transformações matemáticas para melhorar relações lineares

# 1. Preparar as transformações da área
# 2. Para cada transformação: calcular correlação, R², plotar
# 3. Comparar resultados e escolher a melhor
# 4. Repetir processo com variável de distância (bonus)

# Estrutura sugerida:
# transformations_results = {
#     'original': {'correlation': 0.0, 'r2': 0.0},
#     'log': {'correlation': 0.0, 'r2': 0.0},
#     'sqrt': {'correlation': 0.0, 'r2': 0.0},
#     'squared': {'correlation': 0.0, 'r2': 0.0}
# }

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import matplotlib.pyplot as plt

print("Implemente suas transformações aqui!")
print("Tempo estimado: 8 minutos")

---

## 📝 **Conclusão da Parte 1**

Nesta primeira parte da Aula 09, exploramos os **fundamentos da engenharia de features**:

### ✅ **O que aprendemos:**
- Importância da engenharia de features para melhorar modelos de ML
- Como realizar análise exploratória sistemática para identificar oportunidades
- Transformações matemáticas básicas e seus efeitos:
  - **Logarítmica**: Lineariza relações exponenciais, comprime valores altos
  - **Raiz quadrada**: Suaviza relações não-lineares moderadas
  - **Quadrática**: Amplifica diferenças em valores altos
  - **Inversa**: Transforma distâncias em proximidades
- Interpretação e uso do R² para avaliar transformações

### 📊 **Principais Insights:**
- Relações não-lineares são comuns em dados reais e podem ser linearizadas
- Transformações inversas são especialmente úteis para variáveis de distância
- A escolha da transformação depende do contexto e da natureza dos dados
- Mesmo melhorias pequenas no R² (ex: 0.225 → 0.243) são valiosas em contextos multivariados

### 🔧 **Técnicas Aplicadas:**
- Análise de correlação antes e depois das transformações
- Visualização com scatter plots e linhas de tendência
- Comparação sistemática de diferentes transformações
- Interpretação de métricas para escolha da melhor transformação

### 📚 **Checklist de Aprendizado:**
- [ ] Sei identificar variáveis candidatas a transformação através de EDA
- [ ] Consigo aplicar transformações log, sqrt, square e inverse
- [ ] Entendo como interpretar o R² e correlação
- [ ] Posso visualizar e comparar o efeito de diferentes transformações
- [ ] Sei escolher a transformação mais apropriada para cada variável

---

## 🚀 **Próxima Parte**

Na **Parte 2 - Técnicas Avançadas**, você aprenderá:
- Features polinomiais e interações complexas
- Target encoding para variáveis categóricas
- Engenharia de features temporais e cíclicas
- Features geoespaciais avançadas
- Avaliação e validação das features criadas


---